# Entropy and KLD for pseudo semantic loss

Canonical detox notebook for the entropy/KLD sanity check. The saved output below was computed with `MODEL_NAME = "hf_models/psl_improve_spaces/step_1150_2.0_5e-05/"`, i.e. the finetuned pseudo-SL checkpoint. Set `MODEL_NAME = "gpt2"` and rerun the setup plus estimator cell for the base GPT-2 comparison.


In [1]:
import os

os.environ["TRANSFORMERS_CACHE"] = "/space/ahmedk/cache/"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import itertools

import torch
import torch._dynamo as dynamo
from transformers import GPT2LMHeadModel, GPT2Tokenizer

dynamo.config.cache_size_limit = 10000

device = "cuda"


In [3]:
MODEL_NAME = "hf_models/psl_improve_spaces/step_1150_2.0_5e-05/"
# MODEL_NAME = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME,
    pad_token_id=tokenizer.eos_token_id,
).cuda()


In [4]:
def create_constraint():
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2", use_fast=True)

    with open("bad_words.txt") as f:
        lines = f.readlines()

    data = []
    for line in lines:
        data.append(line.rstrip())
        data.append(" " + line.rstrip())

    words = tokenizer(data)["input_ids"]
    unique_words = torch.tensor(list(itertools.chain.from_iterable(words))).unique().tolist()

    global num_tokens
    num_tokens = len(unique_words) + 1

    global token_idx
    token_idx = num_tokens - 1

    tokenid2varid = {word: i for i, word in enumerate(unique_words)}

    global idxmap
    idxmap = torch.full((50257,), num_tokens - 1)
    idxmap[list(tokenid2varid.keys())] = torch.tensor(list(tokenid2varid.values()))

    print("Number of tokens", num_tokens)


create_constraint()
num_classes = 1200
batch_size = 1


Number of tokens 871


In [9]:
# Compute entropy of local_pseudolikelihood

@torch.jit.script
def log1mexp(x):
    lt = (x < 0.6931471805599453094).logical_and(x > 0)
    gt = x >= 0.6931471805599453094
    res = torch.empty_like(x)
    res[lt] = torch.log(-torch.expm1(-x[lt]))
    res[gt] = torch.log1p(-torch.exp(-x[gt]))
    res = res.masked_fill_(x == 0, -float('inf'))
    return res

entropy = []
for i in range(10):
    seq_len = 10
    batch_size = 1
    sample = model.generate(do_sample=True, max_length=10, num_return_sequences=1, top_k=50257, output_scores=False, return_dict_in_generate=False).cpu()
    attn_mask = torch.ones((1, 10))
    idxmap = idxmap.cpu()

    with torch.no_grad():

        # We'll only consider the top-k tokens under
        # the pseudolikelihood distribution centered
        # around the samples
        # Output Shape: [batch_size, seq_len, k]
        logits_orig = model(sample.cuda())['logits'].cpu()
        logits = logits_orig.sort(descending=True)
        topk_tokens = logits.indices[:, :, :num_classes-1]
        topk_tokens = torch.cat((topk_tokens, sample.unsqueeze(-1)), dim=-1)

        # Shape: [batch_size, seq_len, k, seq_len)
        samples = sample.unsqueeze(1).unsqueeze(1).repeat(1, seq_len, num_classes, 1)

        # We want to modify the samples such that we
        # have batch_size*seq_len*k samples, where for
        # each sample and each position we try one of
        # k tokens
        samples[torch.arange(batch_size).unsqueeze(-1).unsqueeze(-1),
                torch.arange(seq_len).unsqueeze(-1).unsqueeze(0),
                torch.arange(num_classes).unsqueeze(0).unsqueeze(0),
                torch.arange(seq_len).unsqueeze(-1).unsqueeze(0)] = topk_tokens

        # We want to batch all these sample for a single
        # model evaluation
        samples = samples.reshape(-1, seq_len)

        # Compute the likelihood of expanded samples
        log_probs = model(samples.cuda(), attention_mask=attn_mask.unsqueeze(1).\
                unsqueeze(1).expand(batch_size, seq_len, num_classes, seq_len).\
                reshape(-1, seq_len).cuda())['logits']
        log_probs = log_probs.log_softmax(dim=-1)

        # Compute the loglikelihood of each sample
        log_probs = log_probs.gather(-1, samples.cuda().unsqueeze(-1)).squeeze().sum(-1)

        del samples

        # Compute pseudolikelihoods
        lit_weights = log_probs
        lit_weights = lit_weights.view(batch_size, seq_len, num_classes)
        lit_weights = lit_weights - lit_weights.logsumexp(-1, keepdim=True)
        # Correctness Check: assert torch.close(lit_weights.logsumexp(-1).exp(), 1)

        # tmp is going to hold our literal weights
        tmp = torch.full((batch_size, seq_len, 50257), -float('inf'), device='cuda')
        tmp[torch.arange(batch_size).unsqueeze(1).unsqueeze(1),
            torch.arange(seq_len).unsqueeze(-1), topk_tokens] = lit_weights

        m = torch.distributions.categorical.Categorical(probs=tmp.exp().squeeze())
        print("Entropy: ", sum(m.entropy()))
        entropy += [sum(m.entropy())]
        
        samples = m.sample(sample_shape=(1000,))

        nlls = []
        for i, sample in enumerate(samples):
            sample = sample.unsqueeze(0).to(device)
            p_x = model(sample).logits.log_softmax(-1)
            p_x = p_x.gather(-1, sample.cuda().unsqueeze(-1)).squeeze().sum()
            q_x = tmp.gather(-1, sample.cuda().unsqueeze(-1)).squeeze().sum()

            nlls.append((q_x-p_x))
            # nlls.append(p_x-q_x)
        print("KLD:", torch.stack(nlls).mean())

Entropy:  tensor(33.7545, device='cuda:0')
KLD: tensor(6.3434, device='cuda:0')
Entropy:  tensor(35.4558, device='cuda:0')
KLD: tensor(1.2079, device='cuda:0')
Entropy:  tensor(30.8389, device='cuda:0')
KLD: tensor(3.4390, device='cuda:0')
Entropy:  tensor(34.5330, device='cuda:0')
KLD: tensor(8.7540, device='cuda:0')
Entropy:  tensor(41.7026, device='cuda:0')
KLD: tensor(4.0770, device='cuda:0')
Entropy:  tensor(38.2238, device='cuda:0')
KLD: tensor(-3.2505, device='cuda:0')
Entropy:  tensor(34.2602, device='cuda:0')
KLD: tensor(3.0317, device='cuda:0')
Entropy:  tensor(34.7254, device='cuda:0')
KLD: tensor(7.4904, device='cuda:0')
Entropy:  tensor(36.8368, device='cuda:0')
KLD: tensor(1.1648, device='cuda:0')
Entropy:  tensor(39.3222, device='cuda:0')
KLD: tensor(7.8083, device='cuda:0')


In [ ]:
# Summary of the saved output above.
entropy_values = torch.tensor([
    33.7545, 35.4558, 30.8389, 34.5330, 41.7026,
    38.2238, 34.2602, 34.7254, 36.8368, 39.3222,
])
kld_values = torch.tensor([
    6.3434, 1.2079, 3.4390, 8.7540, 4.0770,
    -3.2505, 3.0317, 7.4904, 1.1648, 7.8083,
])

print("mean entropy", entropy_values.mean().item())
print("mean KLD", kld_values.mean().item())
